In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model


In [ ]:
class AgentsState(MessagesState):
    current_agent: str
    transfered_by: str


llm = init_chat_model("openai:gpt-4o")

In [ ]:
def make_agent(prompt: str, tools: list[ToolNode]):
    def agent_node(state: AgentsState):
        llm_with_tools = llm.bind_tools(tools)
        response = llm_with_tools.invoke(
            f"""
        {prompt}

        Conversation history:
        {state.messages}
        """
        )
        return {"messages": [response]}

    agent_builder = StateGraph(AgentsState)

    agent_builder.add_node("agent", agent_node)

    agent_builder.add_node("tools", ToolNode(tools))

    agent_builder.add_edge(START, "agent")

    agent_builder.add_conditional_edges("agent", tools_condition("tools"))
    agent_builder.add_edge("tools", "agent")
    agent_builder.add_edge("agent", END)

    return agent_builder.compile()

In [ ]:
@tool
def handoff_tool(transfer_to: str, transfered_by: str):
    """
    Handoff the another agent.

    Use this tool when the customer speaks a language that you don't understate.


    Possisble values for `transfer_to`:

    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Possisble values for `transfered_by`:
    - `korean_agent`
    - `greek_agent`
    - `spanish_agent`

    Args:
    - transfer_to (str): The agent to transfer the conversation to.
    - transfered_by (str): The agent who is transferring the conversation.
    """


    return Command(
        update={
            transfer_to: {
                "transfer_to": transfer_to,
                "transfered_by": transfered_by,
            }
        },
        goto=transfer_to,
        graph=Command.PARENT
    )

In [ ]:
graph_builder = StateGraph(AgentsState)

graph_builder.add_node(
    "korean_agent",
    make_agent(
        prompt="You are a Korean customer support agent, You only speak and understand Korean.",
        tools=[handoff_tool],
    ),
)

graph_builder.add_node(
    "greek_agent",
    make_agent(
        prompt="You are a Greek customer support agent, You only speak and understand Greek.",
        tools=[handoff_tool],
    ),
)

graph_builder.add_node(
    "spanish_agent",
    make_agent(
        prompt="You are a Spanish customer support agent, You only speak and understand Spanish.",
        tools=[handoff_tool],
    ),
)

graph_builder.add_edge(START, "korean_agent")